<a href="https://colab.research.google.com/github/BukolaEasy/BukolaEasy/blob/main/Copy_of_week_08_wed_demo_sql_global_retail_revenue.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 — End-to-End Business Analysis: The Full Pipeline
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

This is the capstone week. Nothing new is introduced today — instead, everything from
Weeks 1–7 comes together into one piece of work you could actually hand to a leadership team.

By the end of this session, you will be able to:
- Combine JOINs, CTEs, window functions, `CASE WHEN` and date functions into a **single coherent
  business analysis**, rather than a pile of unrelated queries.
- Read a business brief and decide which SQL building blocks answer each question in it.
- Work in your group to produce analysis where **every number is SQL-verified** — the standard your
  Thursday presentation will be marked against.

### Setup — run this cell first

It loads all 8 Olist tables into a file-based SQLite database and connects the `%%sql` magic to it.
Everything below depends on it, so run it before anything else. Because `autopandas` is on, every
`%%sql` result comes back as a pandas DataFrame.

In [1]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71

Mounted at /content/drive
Searching your Google Drive for phase-2-python-sql.zip ...
Unzipping phase-2-python-sql.zip ...
Data folder: /content/olist_data/phase-2-python-sql
Loaded orders: 99,441 rows
Loaded customers: 99,441 rows
Loaded order_items: 112,650 rows
Loaded order_payments: 103,886 rows
Loaded order_reviews: 99,224 rows
Loaded products: 32,951 rows
Loaded sellers: 3,095 rows
Loaded product_category_translation: 71 rows

Database ready.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.1/95.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.8/192.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 812.9/812.9 kB 20.6 MB/s eta 0:00:00


## Why this matters — the Business Brief

For seven weeks you have answered questions somebody else wrote for you. Today the shape of the work
changes. Here is the brief, exactly as your group will be marked on it:

> *You are a data analyst presenting to Olist's leadership team. They want to understand:
> (1) How has the platform grown? (2) Which sellers drive the most value? (3) Which product
> categories are the real revenue drivers? (4) How does delivery performance affect customer
> satisfaction? Your analysis must be SQL-only and all numbers must be verified.*

Notice what the brief does **not** do: it does not tell you which tables to use, which clause to
reach for, or what the answer looks like. That is the job now. The platform you are reporting on
holds **99,441 orders**, of which **96,478** were delivered and **625** cancelled — and across all
those orders, **R$15,843,553.24** of goods and freight changed hands. Leadership will not read your
SQL. They will read four or five numbers and a sentence about what each one means, and if a number
is wrong, everything built on top of it is wrong too.

The three analyses below are worked demonstrations of that standard. Each one is a full pipeline:
a `WITH` block that shapes the data, an outer `SELECT` that presents it, and a business reading of
the result. Unlike previous weeks, these queries return **many rows** — there is no single tick-box
answer. Your job while they run is to look at the output and say out loud what it tells leadership.

## Group Exercise — the brief, split four ways

⏱ the rest of this session · in your project groups

The brief above is the assessed piece of work, and from here on you tackle it **as a group**. Split
the four leadership questions between you, one owner each, and agree now who is taking which:

1. **How has the platform grown?** — year on year, in orders, revenue, basket size and customers
   reached. Owner needs `strftime` on the TEXT purchase timestamp plus `GROUP BY` on the year.
2. **Which sellers drive the most value?** — rank sellers by revenue, then band them into tiers.
   Owner needs a `WITH` block to total each seller first, then `CASE WHEN` (or a window function)
   over that result.
3. **Which product categories are the real revenue drivers?** — top categories by revenue, with each
   one's share of the platform total. Owner needs `SUM` inside a CTE and a window `SUM(...) OVER ()`
   for the share, or the verified platform total of **R$15,843,553.24** as the denominator.
4. **How does delivery performance affect customer satisfaction?** — join delivery timing to review
   scores and describe the relationship in one sentence.

Between the four of you, that is every tool of the last seven weeks in play at once: **JOINs** to
bring `orders`, `order_items`, `order_payments`, `order_reviews`, `customers`, `sellers` and
`products` together; **CTEs** to shape the data one readable step at a time; **window functions**
to rank and to compute shares without a second pass; **CASE WHEN** to turn raw numbers into tiers
and flags; and **date functions** to put everything on a timeline.

**The rule that overrides everything else: every number you present must be SQL-verified.** Not
estimated, not remembered from a slide, not taken from DeepSeek without running it. If you quote a
figure, you must be able to show the query that produced it and a second query that agrees with it.
A useful anchor: your own totals should reconcile to **99,441** orders, of which **96,478** were
delivered.

Work through the three analyses below together — they are your shared toolkit. Each one is a
complete pipeline you can lift the *shape* of (not the numbers) into whichever of the four questions
you own. Then take them apart: ask what the CTE is doing, why the join is in that order, and where
the grain could have gone wrong.

2. **Which sellers drive the most value?** — rank sellers by revenue, then band them into tiers.
   Owner needs a `WITH` block to total each seller first, then `CASE WHEN` (or a window function)
   over that result.

In [13]:
%%sql
WITH seller_revenue AS (
SELECT s.seller_id,
       SUM(oi.price + oi.freight_value) AS total_revenue
FROM order_items AS oi
JOIN sellers AS s ON oi.seller_id = s.seller_id
GROUP BY s.seller_id
)
SELECT seller_id, total_revenue,
      CASE WHEN total_revenue >= 100000 THEN 'Top Seller'
           WHEN total_revenue >= 50000 THEN 'High Performer'
           WHEN total_revenue >= 10000 THEN 'Mid Seller'
           ELSE 'Standard'
      END AS seller_tier,
     RANK () OVER (ORDER BY total_revenue DESC) AS seller_rank
FROM seller_revenue

,seller_id,total_revenue,seller_tier,seller_rank
0,4869f7a5dfa277a7dca6462dcf3b52b2,249640.70,Top Seller,1
1,7c67e1448b00f6e969d365cea6b010ab,239536.44,Top Seller,2
2,53243585a1d6dc2643021fd1853d8905,235856.68,Top Seller,3
3,4a3ca9315b744ce9f8e9374361493884,235539.96,Top Seller,4
4,fa1c13f2614d7b5c4749cbc52fecda94,204084.73,Top Seller,5
...,...,...,...,...
3090,702835e4b785b67a084280efca355756,18.56,Standard,3091
3091,4965a7002cca77301c82d3f91b82e1a9,16.36,Standard,3092
3092,1fa2d3def6adfa70e58c276bb64fe5bb,15.90,Standard,3093
3093,77128dec4bec4878c37ab7d6169d6f26,15.22,Standard,3094


In [19]:
%%sql
WITH seller_revenue AS (
SELECT s.seller_id,
       SUM(oi.price + oi.freight_value) AS total_revenue
FROM order_items AS oi
--JOIN sellers AS s ON oi.seller_id = s.seller_id
--GROUP BY s.seller_id
),
ranked_tiers AS (
    SELECT seller_id, total_revenue,
      CASE WHEN total_revenue >= 100000 THEN 'Top Seller'
           WHEN total_revenue >= 50000 THEN 'High Performer'
           WHEN total_revenue >= 10000 THEN 'Mid Seller'
           ELSE 'Standard'
      END AS seller_tier,
     RANK () OVER (ORDER BY total_revenue DESC) AS seller_rank
FROM seller_revenue
)
SELECT seller_id, total_revenue, seller_tier, seller_rank
FROM ranked_tiers
WHERE seller_tier = 'Top Seller'

RuntimeError: (sqlite3.OperationalError) no such column: s.seller_id
[SQL: WITH seller_revenue AS (
SELECT s.seller_id,
       SUM(oi.price + oi.freight_value) AS total_revenue
FROM order_items AS oi

),
ranked_tiers AS (
    SELECT seller_id, total_revenue,
      CASE WHEN total_revenue >= 100000 THEN 'Top Seller'
           WHEN total_revenue >= 50000 THEN 'High Performer'
           WHEN total_revenue >= 10000 THEN 'Mid Seller'
           ELSE 'Standard'
      END AS seller_tier,
     RANK () OVER (ORDER BY total_revenue DESC) AS seller_rank
FROM seller_revenue
)
SELECT seller_id, total_revenue, seller_tier, seller_rank
FROM ranked_tiers
WHERE seller_tier = 'Top Seller']
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [4]:
%%sql
SELECT *
FROM sellers

,seller_id,seller_zip_code_prefix,seller_city,seller_state
0,3442f8959a84dea7ee197c632cb2df15,13023,campinas,SP
1,d1b65fc7debc3361ea86b5f14c68d2e2,13844,mogi guacu,SP
2,ce3ad9de960102d0677a81f5d0bb7b2d,20031,rio de janeiro,RJ
3,c0f3eea2e14555b6faeea3dd58c1b1c3,4195,sao paulo,SP
4,51a04a8a6bdcb23deccc82b0b80742cf,12914,braganca paulista,SP
...,...,...,...,...
3090,98dddbc4601dd4443ca174359b237166,87111,sarandi,PR
3091,f8201cab383e484733266d1906e2fdfa,88137,palhoca,SC
3092,74871d19219c7d518d0090283e03c137,4650,sao paulo,SP
3093,e603cf3fec55f8697c9059638d6c8eb5,96080,pelotas,RS


In [5]:
%%sql
SELECT *
FROM order_items

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


In [6]:
%%sql
SELECT *
FROM order_payments

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45
...,...,...,...,...,...
103881,0406037ad97740d563a178ecc7a2075c,1,boleto,1,363.31
103882,7b905861d7c825891d6347454ea7863f,1,credit_card,2,96.80
103883,32609bbb3dd69b3c066a6860554a77bf,1,credit_card,1,47.77
103884,b8b61059626efa996a60be9bb9320e10,1,credit_card,5,369.54


## Analysis 1 — Platform growth: 2017 vs 2018

The first question in the brief is *how has the platform grown?* — and "grown" is vague, so you make
it specific. Growth has at least four faces here: more orders placed, more money taken, a larger
average basket, and more distinct customers reached. A business can grow on one of those and shrink
on another, which is exactly why you report them side by side instead of picking one.

The query below uses a CTE (`yearly_summary`, Week 7) to build one row per year, `strftime('%Y', ...)`
(Week 4) to pull the year out of the TEXT purchase timestamp, a JOIN (Week 3) to reach payment values,
and `COUNT(DISTINCT ...)` (Week 2) so that an order with two payment rows is still counted once.

One deliberate choice: the `WHERE` clause keeps only 2017 and 2018. The dataset also contains
**329 orders from 2016** — a partial year of pilot data against **45,101** orders in 2017 and
**54,011** in 2018. Including 2016 would produce a spectacular, meaningless growth rate. Excluding a
year is a judgement call, and a judgement call belongs in your presentation, not hidden in a `WHERE`.

In [ ]:
%%sql
-- Analysis 1 — platform growth, one row per year.
WITH yearly_summary AS (
    SELECT strftime('%Y', o.order_purchase_timestamp) AS year,
           COUNT(DISTINCT o.order_id) AS total_orders,
           ROUND(SUM(op.payment_value), 2) AS total_revenue,
           ROUND(AVG(op.payment_value), 2) AS avg_order_value,
           COUNT(DISTINCT o.customer_id) AS unique_customers
    FROM orders o
    JOIN order_payments op ON o.order_id = op.order_id
    WHERE strftime('%Y', o.order_purchase_timestamp) IN ('2017', '2018')
    GROUP BY year
)
SELECT year, total_orders, total_revenue, avg_order_value, unique_customers
FROM yearly_summary
ORDER BY year

### Reading the result

Take the two rows side by side and read them as leadership would:

- **`total_orders`** — the volume story, and your first verification. It should read **45,101** for
  2017 and **54,011** for 2018, matching the `orders` table exactly. That match is worth a moment's
  thought rather than a shrug: this query reaches `order_payments` through an inner JOIN, and one
  order in the whole dataset has no payment row at all (99,440 distinct `order_id` values across
  103,886 payment rows). An inner JOIN silently drops that order — it simply happens to be a 2016
  order, which the `WHERE` clause excludes anyway. Get into the habit of asking *which rows could
  this JOIN have dropped?* before you quote a total to anybody.
- **`total_revenue`** — the money story, and the one leadership will quote back at you.
- **`avg_order_value`** — read this one with care. It is `AVG(op.payment_value)`, which averages
  **payment rows**, not orders. An order settled in two instalments contributes two smaller rows.
  The column is *named* `avg_order_value`, but that is not quite what it measures. We fix this
  properly in the "Going deeper" section — and the fix is a strong point to make in a presentation.
- **`unique_customers`** — reach. In this dataset `customer_id` is unique per order, so compare it to
  `total_orders` and ask what that tells you about repeat buying.

> **Talking point for your group:** if orders grew faster than revenue, average basket size fell.
> Which one would you lead the slide with, and what would you recommend because of it?

## Analysis 2 — Delivery performance by state

The brief's fourth question links delivery to satisfaction. Before you can connect the two, you have
to measure delivery itself — and "delivery performance" is really three numbers: how many orders a
state received, how long they took, and how often the promise made at checkout was broken.

Three Week-4 ideas do the heavy lifting inside the CTE. `julianday(...) - julianday(...)` converts
two TEXT timestamps into a number of days, because SQLite has no date type and no `DATEDIFF`.
`CASE WHEN ... THEN 1 ELSE 0 END` inside a `SUM` turns a yes/no question into a count — the standard
way to count "how many rows satisfy this condition" alongside other aggregates. And
`order_delivered_customer_date > order_estimated_delivery_date` is the definition of *late*: the
order arrived after the date the customer was promised.

The outer query then adds two things worth pausing on. `late_count * 100.0 / delivered_orders` forces
REAL division with the `* 100.0` — write `100` instead and SQLite performs integer division, handing
you `0` for every state. And `AVG(avg_days) OVER ()` is a window function (Week 5) with an **empty**
`OVER ()`: no `PARTITION BY`, no `ORDER BY`, so the window is the whole result set. That puts a
benchmark on every row, so each state can be read against the national picture instead of in isolation.

In [ ]:
%%sql
-- Analysis 2 — delivery performance per customer state, ranked fastest first.
WITH delivery_stats AS (
    SELECT c.customer_state,
           COUNT(*) AS delivered_orders,
           ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)), 1) AS avg_days,
           SUM(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS late_count
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY c.customer_state
)
SELECT customer_state,
       delivered_orders,
       avg_days,
       late_count,
       ROUND(late_count * 100.0 / delivered_orders, 1) AS late_pct,
       ROUND(AVG(avg_days) OVER (), 1) AS national_avg_days
FROM delivery_stats
ORDER BY avg_days ASC
LIMIT 10

### Reading the result

Ten rows, fastest states first. Three habits to build here:

- **Always read `avg_days` next to `delivered_orders`.** A state with a few hundred deliveries can top
  the table on luck alone. São Paulo carries **41,746** customers — a state that size moving its
  average is a real operational change; a small state moving is noise.
- **`national_avg_days` repeats on every row**, which surprises people the first time. That is what an
  empty `OVER ()` does: one value computed across all rows of the result, attached to each. Now look
  at the value closely — it is noticeably **higher than the 12.6 days** that a delivery takes on
  average across the whole dataset. That is not an error, and spotting why is the most valuable thing
  in this cell: `AVG(avg_days) OVER ()` averages the **state averages**, giving tiny Roraima exactly
  the same weight as São Paulo. The 12.6-day figure averages **orders**, and most orders are in the
  fast southeastern states. Same data, two legitimate questions: *how long does a typical order take?*
  (12.6 days) versus *how long does a typical state wait?* (the higher number on screen). Say which
  one you mean on your slide, because they support different recommendations.
- **`avg_days` and `late_pct` are different questions.** Slow is not the same as late: a state can be
  slow and rarely late (the estimate was generous) or fast and often late (the estimate was
  optimistic). Platform-wide, **7,826** deliveries arrived after their estimated date — **8.1%** of
  all delivered orders. Watch for a state in the top ten on speed that still has a high `late_pct`;
  that is a promise-setting problem, not a logistics problem, and it is a genuinely interesting
  finding to put in front of leadership.

## Analysis 3 — Does delivery speed explain review scores?

Now the two halves of the brief's fourth question meet. Instead of grouping by state, group by
`review_score` — 1 to 5 — and ask what delivery looked like for the orders behind each score. If slow
delivery drives bad reviews, the pattern will be visible without any statistics: `avg_delivery_days`
will fall as the score rises, and `late_pct` will fall with it.

This is a good moment to notice how you are using the tables. `order_reviews` holds many rows per
order (99,224 reviews across 98,673 distinct orders — a few orders were reviewed twice), so a
`COUNT(*)` here counts **reviews**, not orders, which is why the column is honestly named
`order_count` only because each row is one review of one delivered order. The reason this query is
still safe is that it joins **one** many-per-order table and no more. The moment you add
`order_items` or `order_payments` alongside `order_reviews`, rows multiply and every aggregate
inflates — the single most common way a capstone number comes out wrong. More on that below.

The `WHERE` clause keeps only delivered orders with a real delivery date, because an order that never
arrived has no delivery duration to average.

In [ ]:
%%sql
-- Analysis 3 — delivery experience broken down by the review score it earned.
WITH delivery_and_review AS (
    SELECT r.review_score,
           ROUND(AVG(julianday(o.order_delivered_customer_date) - julianday(o.order_purchase_timestamp)), 1) AS avg_delivery_days,
           COUNT(*) AS order_count,
           SUM(CASE WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date THEN 1 ELSE 0 END) AS late_count
    FROM orders o
    JOIN order_reviews r ON o.order_id = r.order_id
    WHERE o.order_status = 'delivered'
      AND o.order_delivered_customer_date IS NOT NULL
    GROUP BY r.review_score
)
SELECT review_score, avg_delivery_days, order_count,
       ROUND(late_count * 100.0 / order_count, 1) AS late_pct
FROM delivery_and_review
ORDER BY review_score

### Reading the result

Five rows, one per star rating. Read down the `avg_delivery_days` column first, then down `late_pct`,
and describe the shape in one sentence — that sentence is your slide.

For context: across the platform the overall average review score is **4.09**, and **57,328** reviews —
**57.8%** of all reviews — are 5-star. The 5-star `order_count` on screen is a little below 57,328,
and you should be able to explain the gap without looking it up: this query counts only reviews
attached to a **delivered** order with a real delivery date. The bulk of the table still sits at the
happy end, and the 1-star row, though smaller, is where the operational pain lives.

> **Two cautions worth saying out loud in your presentation.** First, this shows *association*, not
> proof of cause: slow orders may also be the expensive, bulky or far-away ones, and any of those
> could be what actually annoyed the customer. Second, every row here is a **delivered** order.
> Customers whose orders never arrived are excluded entirely — and they are unlikely to be the happy
> ones. Naming a limitation before your audience finds it is what "business insight quality" means on
> the marking rubric.

---
## 🤖 Using DeepSeek this week

This week DeepSeek is genuinely useful, because capstone queries are long and you will be writing
them under time pressure. It is also the week where trusting it blindly costs you the most marks:
a fabricated number in a leadership presentation is worse than no number.

Use the same prompt-then-verify protocol you have used since Week 4, with one addition for capstone
work — **anchor every AI-drafted query to a number you already trust.**

1. **Prompt with the schema and the grain.** "Using SQLite tables `orders(order_id, customer_id,
   order_status, order_purchase_timestamp, order_delivered_customer_date,
   order_estimated_delivery_date)` and `customers(customer_id, customer_state)`, write a query giving
   average delivery days per state for delivered orders. Dates are TEXT — use `julianday`."
2. **Read it before you run it.** Does it join two of `order_items` / `order_payments` /
   `order_reviews`? Does it divide integers without `* 1.0`? Does it use `= NULL`? Those are the three
   errors DeepSeek makes most often on this dataset.
3. **Verify against a known total.** Ask the AI's query to reproduce a figure from the curriculum you
   already trust. If it cannot reproduce a number you know, do not trust the number you don't.

The cell below is step 3 in its purest form: the platform overview KPIs, every value verified. Run
this in your group before your presentation — if these four numbers come out right, your database is
loaded correctly and your baseline is sound.

In [ ]:
%%sql
-- Step 3 of the protocol: the anchor query. Every value here is verified —
-- if your notebook reproduces these, your baseline is trustworthy.
SELECT
    COUNT(DISTINCT order_id) AS total_orders,                                          -- Expected: 99,441
    SUM(CASE WHEN order_status = 'delivered' THEN 1 ELSE 0 END) AS delivered,          -- Expected: 96,478
    SUM(CASE WHEN order_status = 'canceled' THEN 1 ELSE 0 END) AS canceled,            -- Expected: 625
    ROUND(SUM(CASE WHEN order_status = 'delivered' THEN 1.0 ELSE 0 END) / COUNT(*) * 100, 1) AS delivered_pct  -- Expected: 97.0
FROM orders

## Going deeper — average order value, at the right grain

Return to the caveat from Analysis 1. `AVG(op.payment_value)` averages payment **rows**. The
`order_payments` table holds **103,886** rows but only **99,440** distinct orders — **2,961** orders
were settled with more than one payment (a voucher plus a card, or a card split into instalments).
Each of those contributes several small rows to the average instead of one whole order value, which
pulls the "average order value" down.

The fix is the pattern that will save your capstone more than any other: **collapse the many-per-order
table to one row per order in a CTE first, then aggregate.** `order_totals` below does exactly that —
`SUM(payment_value) GROUP BY order_id` gives one true order value per order. Only then do we average.

Run it, then compare `true_avg_order_value` with the `avg_order_value` from Analysis 1. The gap
between those two numbers is the cost of getting grain wrong, and explaining it is worth real marks.

In [ ]:
%%sql
-- Payments are at PAYMENT grain: 103,886 rows, only 99,440 distinct orders.
-- Collapse to ONE row per order first, then average.
WITH order_totals AS (
    SELECT order_id,
           SUM(payment_value) AS order_value
    FROM order_payments
    GROUP BY order_id
)
SELECT COUNT(*) AS orders_with_payment,                 -- Expected: 99,440
       ROUND(SUM(order_value), 2) AS total_paid,
       ROUND(AVG(order_value), 2) AS true_avg_order_value
FROM order_totals

## Common mistakes

**Mistake 1 — joining two many-per-order tables and then aggregating.** This is *the* capstone killer.
`order_items` (9,803 orders have more than one row), `order_payments` (2,961 do) and `order_reviews`
(547 do) are all many-per-order. Join any two of them and SQLite hands you every combination: an order
with 3 items and 2 payments becomes 6 rows, and every `SUM`, `AVG` and `COUNT(*)` built on it is
inflated. Nothing errors. The query runs, the number looks plausible, and it is wrong. The fix is
always the same: pre-aggregate the extra table to one row per `order_id` in a CTE, then join — or,
better, notice that you only needed one of them in the first place.

**Mistake 2 — integer division.** `late_count * 100 / delivered_orders` returns `0` for every state,
because SQLite divides two integers into an integer. Write `* 100.0` — as Analysis 2 does — and you
get a real percentage.

**Mistake 3 — `= NULL`.** Analysis 2 and 3 both filter with `order_delivered_customer_date IS NOT NULL`.
Written as `!= NULL` it matches nothing at all, silently, and your delivered-order counts collapse.

The cell below shows the fan-out mistake as a comment (never run a query that produces a wrong number
you might copy), then runs the correct GMV query — the one whose values your Thursday presentation
must reproduce.

In [ ]:
%%sql
-- ── COMMON MISTAKE — join fan-out ───────────────────────────────────
-- WRONG — joins TWO many-per-order tables, then sums money:
--   SELECT ROUND(SUM(oi.price), 2) AS product_revenue
--   FROM orders o
--   JOIN order_items    oi ON o.order_id = oi.order_id
--   JOIN order_payments op ON o.order_id = op.order_id
--   -> every item row is repeated once per payment row. No error, inflated total.
--
-- CORRECT — the money question only ever needed order_items, at item grain:
SELECT ROUND(SUM(price), 2) AS product_revenue,             -- Expected: 13,591,643.70
       ROUND(SUM(freight_value), 2) AS freight_revenue,     -- Expected: 2,251,909.54
       ROUND(SUM(price + freight_value), 2) AS total_gmv    -- Expected: 15,843,553.24
FROM order_items

## Mini-challenge — your group's fifth analysis

⏱ ~10 min · in your groups

Your Thursday presentation needs **five** analyses: four are set by the brief (platform KPIs, top 5
categories by revenue, seller tiers, delivery vs reviews) and the fifth is yours to choose. Start it now.

Pick one business question this dataset can actually answer, and write the query for it. Some
starting points, though a question of your own is better:

- Which payment type carries the highest average order value, and does that change by year?
- Which product categories have the worst late-delivery rate — and are they also the lowest-rated?
- How concentrated is the platform? What share of revenue comes from the top 10 sellers?
- Which month was the platform's best, and can you explain the spike? (November 2017 has **7,544**
  orders — there is a reason.)

Rules, so that the work counts on Thursday:
1. **SQL only.** No pandas beyond displaying the result.
2. **Use a CTE** if the query needs more than one step — readability carries 10 of the 40 marks.
3. **Watch the grain.** If your question touches two of `order_items` / `order_payments` /
   `order_reviews`, pre-aggregate one of them first.
4. **Sanity-check your number** against something you already trust before you put it on a slide.

Replace the placeholder below with your query.

In [ ]:
%%sql
-- ⏱ ~10 min — your turn! Replace the placeholder below with your group's fifth analysis.
-- Sketch the steps in comments first, then write the CTE, then the outer SELECT.
SELECT 'write your group query here' AS todo

## Session Summary

Eight weeks of SQL, assembled into one pipeline. Everything you used today, and where it came from:

| Skill | Week | What it contributed today |
|---|---|---|
| `SELECT` / `WHERE` / `ORDER BY` / `LIMIT` | 1 | Filtering to delivered orders; showing the top 10 states |
| `GROUP BY`, aggregates, `COUNT(DISTINCT ...)` | 2 | One row per year, per state, per review score — without double-counting |
| `JOIN` | 3 | Reaching customer state, payment value and review score from `orders` |
| `CASE WHEN`, `strftime`, `julianday` | 4 | Counting late deliveries; pulling the year from a TEXT date; measuring delivery days |
| Window functions — `AVG(...) OVER ()` | 5 | A national benchmark attached to every state row |
| Multi-table joins and grain discipline | 6 | Knowing which tables may safely meet in one query |
| CTEs (`WITH`) | 7 | Naming each step so the analysis reads top-to-bottom |

Three things to carry into Thursday:

1. **A business analysis is a story with numbers in it**, not a list of queries. Every result today was
   followed by a sentence about what it means.
2. **Grain before aggregation.** Collapse many-per-order tables to one row per order in a CTE before
   you `SUM` or `AVG` across them.
3. **Verify everything.** Anchor your work to the figures you know: 99,441 orders, 96,478 delivered,
   97.0% delivered, R$15,843,553.24 total GMV.

---
**Coming up Thursday — Presentation Preparation**: your group prepares and delivers a 10-minute
SQL-driven business analysis. Five analyses (platform overview KPIs, top 5 product categories by
revenue, seller tier breakdown using the Week 7 CTE, delivery performance and its relationship to
review scores, plus the one you chose above), in SQL only, with every KPI matching its verified value.
Marks: SQL correctness 15, business insight quality 15, query readability and structure 10 — 40 total.